# CNN/DailyMail text preprocessing

This notebook performs text preprocessing only. It loads the raw dataset created by `download_dataset.ipynb`, cleans the article and reference summary text conservatively, recalculates article word counts, and saves the processed dataset.

The processed dataset contains exactly these columns:

- `article_id`
- `article` — the processed article text
- `summary` — the processed reference summary
- `article_word_count`

Aspect extraction, prompt construction, and training-target construction are intentionally deferred.

## Setup

In [1]:
import json
import os
import re
import unicodedata
from pathlib import Path

import pandas as pd

RAW_PATH = Path("data/raw/cnn_dailymail_raw.jsonl")
PROCESSED_DIR = Path("data/processed")
OUTPUT_PATH = PROCESSED_DIR / "cnn_processed.jsonl"

REQUIRED_RAW_COLUMNS = {"article_id", "article", "summary"}
PROCESSED_COLUMNS = [
    "article_id",
    "article",
    "summary",
    "article_word_count",
]
WORD_RE = re.compile(r"\b\w+(?:[’'-]\w+)*\b", flags=re.UNICODE)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raw input:        {RAW_PATH}")
print(f"Processed output: {OUTPUT_PATH}")
print(f"Output columns:   {PROCESSED_COLUMNS}")

Raw input:        data\raw\cnn_dailymail_raw.jsonl
Processed output: data\processed\cnn_processed.jsonl
Output columns:   ['article_id', 'article', 'summary', 'article_word_count']


## Load the raw dataset

The raw JSONL is validated line by line. This notebook does not download or resample the dataset.

In [2]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Could not find '{RAW_PATH}'. Run download_dataset.ipynb first."
    )

raw_records = []
with RAW_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number} of '{RAW_PATH}'."
            ) from error

        missing_columns = REQUIRED_RAW_COLUMNS - set(record)
        if missing_columns:
            raise ValueError(
                f"Line {line_number} is missing columns: {sorted(missing_columns)}"
            )
        raw_records.append(record)

if not raw_records:
    raise ValueError(f"'{RAW_PATH}' contains no records.")

raw_df = pd.DataFrame(raw_records)
print(f"Loaded {len(raw_df)} raw records")
print("Raw columns:", raw_df.columns.tolist())
raw_df[["article_id", "article_word_count", "summary"]].head()

Loaded 300 raw records
Raw columns: ['article_id', 'article', 'summary', 'article_word_count']


,article_id,article_word_count,summary
0,1640e830cd77ab1e2ba2ea3ca824a7f1f5520d7a,491,Shots rang out on third floor of Renown Region...
1,20ca3cfbdefa3591780fe093cd278d4774fd6601,431,"Teenage girl, 15, kidnapped by three teenage b..."
2,6b1d3b88824cef966e28c3411a48cee8920d65b7,413,Chetana Guduru found mortally wounded in her m...
3,35a582b2de09140d060227c943d09e46e8be15be,379,Cyliyah Yarbrough died from severe head injuri...
4,1e23fca93af6597473dfc60799cd67ca8e2663ee,400,Pedestrian Roman Dati was hit by the journalis...


## Apply text preprocessing

The cleaning is intentionally conservative:

- normalize Unicode to NFKC;
- standardize line endings;
- remove non-printing control characters;
- collapse repeated whitespace in articles;
- preserve meaningful line boundaries in reference summaries;
- remove records with empty required text;
- remove duplicate IDs and duplicate processed articles; and
- recalculate word count from the processed article.

Case, punctuation, names, dates, and numbers are retained because they carry factual information.

In [3]:
def clean_text(text: str, preserve_newlines: bool = False) -> str:
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)

    if preserve_newlines:
        cleaned_lines = []
        for line in text.split("\n"):
            line = re.sub(r"[ \t]+", " ", line).strip()
            if line:
                cleaned_lines.append(line)
        return "\n".join(cleaned_lines)

    return re.sub(r"\s+", " ", text).strip()

    return re.sub(r"\s+", " ", text).strip()

def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))

def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))

processed_df = raw_df.copy()
processed_df["article_id"] = processed_df["article_id"].apply(clean_text)
processed_df["article"] = processed_df["article"].apply(clean_text)
processed_df["summary"] = processed_df["summary"].apply(
    lambda text: clean_text(text, preserve_newlines=True)
)

records_before_cleaning = len(processed_df)
processed_df = processed_df[
    processed_df["article_id"].astype(bool)
    & processed_df["article"].astype(bool)
    & processed_df["summary"].astype(bool)
].copy()
processed_df = processed_df.drop_duplicates(subset=["article_id"], keep="first")
processed_df = processed_df.drop_duplicates(subset=["article"], keep="first")
processed_df = processed_df.reset_index(drop=True)
processed_df["article_word_count"] = processed_df["article"].apply(count_words)
processed_df = processed_df[PROCESSED_COLUMNS]

print(f"Records before preprocessing: {records_before_cleaning}")
print(f"Records after preprocessing:  {len(processed_df)}")
print(
    "Processed article word-count range:",
    int(processed_df["article_word_count"].min()),
    "to",
    int(processed_df["article_word_count"].max()),
)
processed_df.head()

Records before preprocessing: 300
Records after preprocessing:  300
Processed article word-count range: 96 to 499


,article_id,article,summary,article_word_count
0,1640e830cd77ab1e2ba2ea3ca824a7f1f5520d7a,By . Snejana Farberov . PUBLISHED: . 18:22 EST...,Shots rang out on third floor of Renown Region...,491
1,20ca3cfbdefa3591780fe093cd278d4774fd6601,By . Daily Mail Reporter . PUBLISHED: . 17:53 ...,"Teenage girl, 15, kidnapped by three teenage b...",431
2,6b1d3b88824cef966e28c3411a48cee8920d65b7,By . Mail Online Reporter . PUBLISHED: . 13:24...,Chetana Guduru found mortally wounded in her m...,413
3,35a582b2de09140d060227c943d09e46e8be15be,By . Daily Mail Reporter . PUBLISHED: . 09:00 ...,Cyliyah Yarbrough died from severe head injuri...,379
4,1e23fca93af6597473dfc60799cd67ca8e2663ee,By . Daily Mail Reporter . PUBLISHED: . 12:16 ...,Pedestrian Roman Dati was hit by the journalis...,400


## Save all processed records

The complete output is validated in a temporary file before atomically replacing `data/processed/cnn_processed.jsonl`.

In [4]:
assert list(processed_df.columns) == PROCESSED_COLUMNS
assert processed_df["article_id"].is_unique
assert processed_df["article"].is_unique
assert processed_df["article_id"].str.strip().ne("").all()
assert processed_df["article"].str.strip().ne("").all()
assert processed_df["summary"].str.strip().ne("").all()
assert processed_df.apply(
    lambda row: count_words(row["article"]) == row["article_word_count"],
    axis=1,
).all()

temporary_path = OUTPUT_PATH.with_suffix(OUTPUT_PATH.suffix + ".tmp")
try:
    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in processed_df.to_dict(orient="records"):
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    with temporary_path.open("r", encoding="utf-8") as handle:
        saved_records = [json.loads(line) for line in handle if line.strip()]

    assert len(saved_records) == len(processed_df)
    assert all(list(record) == PROCESSED_COLUMNS for record in saved_records)
    assert all(
        count_words(record["article"]) == record["article_word_count"]
        for record in saved_records
    )

    os.replace(temporary_path, OUTPUT_PATH)
finally:
    if temporary_path.exists():
        temporary_path.unlink()

print(f"Saved {len(saved_records)} processed records to '{OUTPUT_PATH}'")
print("Saved columns:", list(saved_records[0]))

Saved 300 processed records to 'data\processed\cnn_processed.jsonl'
Saved columns: ['article_id', 'article', 'summary', 'article_word_count']
